# Deepfake Audio Detection — Final Pipeline
## MARS Open Projects 2026 — AIML Problem Statement 2

This notebook contains the complete end-to-end pipeline:
1. Dataset loading & statistics
2. Audio preprocessing & LFCC feature extraction
3. Light CNN model architecture
4. Model training (with weighted loss, data augmentation, early stopping)
5. Full evaluation: Accuracy, EER, F1, Confusion Matrix, Per-Class Accuracy


## 1. Setup & Imports


In [ ]:
import os, sys, json, pickle, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score, confusion_matrix
from tqdm import tqdm
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"librosa: {librosa.__version__}")


## 2. Dataset Loading & Statistics

The Fake-or-Real (FoR) dataset — `for-norm` variant (16kHz, mono, silence-trimmed, peak-normalized).


In [ ]:
# Dataset path (adjust to your local structure)
DATA_BASE = 'data/FoR/raw/for-norm/for-norm'
TRAIN_DIR = os.path.join(DATA_BASE, 'training')
VAL_DIR = os.path.join(DATA_BASE, 'validation')
TEST_DIR = os.path.join(DATA_BASE, 'testing')

def count_files(d):
    genuine = len(os.listdir(os.path.join(d, 'real'))) if os.path.exists(os.path.join(d, 'real')) else 0
    fake = len(os.listdir(os.path.join(d, 'fake'))) if os.path.exists(os.path.join(d, 'fake')) else 0
    return genuine, fake

for name, d in [('Training', TRAIN_DIR), ('Validation', VAL_DIR), ('Testing', TEST_DIR)]:
    g, f = count_files(d)
    print(f"{name:12s}: Genuine={g:6d}  Deepfake={f:6d}  Total={g+f:6d}  Ratio=Genuine:{f/(g+f)*100:.1f}% Deepfake:{g/(g+f)*100:.1f}%")

# Check audio properties
import soundfile as sf
for cls in ['real', 'fake']:
    d = os.path.join(TRAIN_DIR, cls)
    fpath = os.path.join(d, os.listdir(d)[0])
    info = sf.info(fpath)
    print(f"\n{cls}: sr={info.samplerate}Hz channels={info.channels} duration={info.duration:.1f}s format={info.format}")


## 3. Audio Preprocessing & LFCC Feature Extraction

### Why LFCC?
Linear Frequency Cepstral Coefficients preserve high-frequency energy better than mel-scale MFCC. Neural TTS/VC systems leave artifacts in high frequencies (4-8 kHz) that mel compression would attenuate.

### Pipeline
1. Resample to 16 kHz mono
2. Silence trim (30 dB threshold)
3. Peak normalize to ±1.0
4. Pad/truncate to 4 seconds
5. Extract LFCC: 60 coefficients + delta + delta-delta = 180-dim, 401 frames


In [ ]:
SAMPLE_RATE = 16000
FIXED_LENGTH = SAMPLE_RATE * 4
LFCC_DIM = 60
LFCC_FRAMES = 401

def preprocess_audio(path, augment=False):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    audio, _ = librosa.effects.trim(audio, top_db=30)
    if len(audio) == 0:
        audio = np.zeros(FIXED_LENGTH)
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    if len(audio) > FIXED_LENGTH:
        audio = audio[:FIXED_LENGTH]
    elif len(audio) < FIXED_LENGTH:
        audio = np.pad(audio, (0, FIXED_LENGTH - len(audio)))
    if augment:
        r = np.random.random()
        if r < 0.3:
            snr_db = np.random.uniform(10, 20)
            sp = np.mean(audio ** 2)
            audio = audio + np.sqrt(sp / (10 ** (snr_db / 10))) * np.random.randn(len(audio))
        if r < 0.5:
            audio = librosa.effects.time_stretch(y=audio, rate=np.random.uniform(0.9, 1.1))
        if r < 0.6:
            audio = librosa.effects.pitch_shift(y=audio, sr=SAMPLE_RATE, n_steps=np.random.uniform(-2, 2))
    return audio

def extract_lfcc(audio):
    win = 400
    if len(audio) < win:
        audio = np.pad(audio, (0, win - len(audio)))
    D = np.abs(librosa.stft(audio, n_fft=512, hop_length=160, win_length=win))
    freqs = librosa.fft_frequencies(sr=SAMPLE_RATE, n_fft=512)
    fb = np.zeros((LFCC_DIM, 257))
    for i in range(LFCC_DIM):
        fb[i] = freqs ** (i + 1)
    fb /= (fb.sum(axis=1, keepdims=True) + 1e-10)
    spec = np.dot(fb, D)
    spec_db = librosa.amplitude_to_db(spec, ref=np.max)
    d1 = librosa.feature.delta(spec_db, width=3)
    d2 = librosa.feature.delta(spec_db, width=3, order=2)
    lfcc = np.vstack([spec_db, d1, d2])
    if lfcc.shape[1] > LFCC_FRAMES:
        lfcc = lfcc[:, :LFCC_FRAMES]
    elif lfcc.shape[1] < LFCC_FRAMES:
        lfcc = np.pad(lfcc, ((0, 0), (0, LFCC_FRAMES - lfcc.shape[1])))
    return lfcc

# Demo: extract features from a sample
demo_path = os.path.join(TRAIN_DIR, 'real', os.listdir(os.path.join(TRAIN_DIR, 'real'))[0])
audio = preprocess_audio(demo_path)
lfcc = extract_lfcc(audio)
print(f"Audio shape: {audio.shape}, LFCC shape: {lfcc.shape} (180 dims x {lfcc.shape[1]} frames)")

# Visualize
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))
ax1.plot(np.linspace(0, 4, len(audio)), audio)
ax1.set_title('Waveform (4 seconds)')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Amplitude')
librosa.display.specshow(lfcc[:60], sr=SAMPLE_RATE, hop_length=160, x_axis='time', ax=ax2)
ax2.set_title('LFCC (60 static coefficients)')
plt.tight_layout(); plt.show()


## 4. PyTorch Dataset


In [ ]:
class LFCCDataset(Dataset):
    def __init__(self, root, augment=False):
        self.samples = []
        self.augment = augment
        for cls, label in [('real', 0), ('fake', 1)]:
            d = os.path.join(root, cls)
            if os.path.exists(d):
                for f in os.listdir(d):
                    if f.endswith('.wav'):
                        self.samples.append((os.path.join(d, f), label))
        genuine = sum(1 for _, l in self.samples if l == 0)
        fake = len(self.samples) - genuine
        self.class_weights = torch.tensor([
            len(self.samples) / (2 * genuine) if genuine > 0 else 1.0,
            len(self.samples) / (2 * fake) if fake > 0 else 1.0
        ])
        print(f"Dataset: {len(self.samples)} samples (Genuine={genuine}, Deepfake={fake}), "
              f"ClassWeights={self.class_weights.tolist()}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        audio = preprocess_audio(path, self.augment)
        lfcc = extract_lfcc(audio)
        return torch.tensor(lfcc, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

# Create datasets
train_ds = LFCCDataset(TRAIN_DIR, augment=True)
val_ds = LFCCDataset(VAL_DIR, augment=False)
test_ds = LFCCDataset(TEST_DIR, augment=False)


## 5. Model Architecture: LCNN (Light CNN)

LCNN with Max-Feature-Map activation suppresses weaker neuron activations within pairs, creating compact representations ideal for anti-spoofing.

**Architecture**: Conv blocks (32→64→128→64→64) → BiLSTM (128) → Dropout (0.5) → FC (256→2)

Parameters: ~606K


In [ ]:
class MaxFeatureMap(nn.Module):
    def forward(self, x):
        half = x.size(1) // 2
        return torch.max(x[:, :half], x[:, half:])

class LCNNBlock(nn.Module):
    def __init__(self, in_c, out_c, ks=3, stride=1, pad=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c * 2, ks, stride, pad)
        self.mfm = MaxFeatureMap()
        self.bn = nn.BatchNorm2d(out_c)

    def forward(self, x):
        return self.bn(self.mfm(self.conv(x)))

class LCNN(nn.Module):
    def __init__(self, input_dim=180, num_classes=2):
        super().__init__()
        self.blocks = nn.Sequential(
            LCNNBlock(1, 32), nn.MaxPool2d(2),
            LCNNBlock(32, 64), nn.MaxPool2d(2),
            LCNNBlock(64, 128), nn.MaxPool2d(2),
            LCNNBlock(128, 64),
            LCNNBlock(64, 64),
        )
        self.lstm = nn.LSTM(64, 128, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # add channel dim: (B, 180, 401) -> (B, 1, 180, 401)
        x = self.blocks(x)   # -> (B, 64, H, W)
        x = x.mean(dim=2)    # spatial average -> (B, 64, W)
        x = x.permute(0, 2, 1)  # (B, W, 64) for LSTM
        x, _ = self.lstm(x)  # -> (B, W, 256)
        x = x[:, -1, :]      # take last timestep
        x = self.dropout(x)
        return self.fc(x)

# Verify model
model = LCNN()
print(f"LCNN parameters: {sum(p.numel() for p in model.parameters()):,}")
x = torch.randn(2, 180, 401)
with torch.no_grad():
    out = model(x)
print(f"Input: {x.shape} -> Output: {out.shape} (2-class logits)")


## 6. Evaluation: EER, Accuracy, F1, Confusion Matrix

EER = Equal Error Rate = the point where False Acceptance Rate equals False Rejection Rate.
A lower EER indicates a more balanced, reliable detector.


In [ ]:
def compute_eer(y_true, y_scores):
    '''Compute Equal Error Rate'''
    genuine = y_scores[y_true == 0]
    spoof = y_scores[y_true == 1]
    if len(genuine) == 0 or len(spoof) == 0:
        return 1.0
    thresholds = np.linspace(0, 1, 1000)
    far = np.array([np.sum(genuine >= t) / len(genuine) for t in thresholds])
    frr = np.array([np.sum(spoof < t) / len(spoof) for t in thresholds])
    diff = np.abs(far - frr)
    eer_idx = diff.argmin()
    return float((far[eer_idx] + frr[eer_idx]) / 2)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_scores = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        scores = torch.softmax(model(x), dim=-1)[:, 1]
        all_labels.append(y.cpu().numpy())
        all_scores.append(scores.cpu().numpy())
    y_true = np.concatenate(all_labels)
    y_scores = np.concatenate(all_scores)
    y_pred = (y_scores >= 0.5).astype(int)

    acc = (y_pred == y_true).mean() * 100
    eer = compute_eer(y_true, y_scores) * 100
    f1 = f1_score(y_true, y_pred) * 100
    ga = (y_pred[y_true == 0] == 0).mean() * 100 if (y_true == 0).sum() > 0 else 0.0
    fa = (y_pred[y_true == 1] == 1).mean() * 100 if (y_true == 1).sum() > 0 else 0.0
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    return {
        'accuracy': acc, 'eer': eer, 'f1_score': f1,
        'genuine_accuracy': ga, 'deepfake_accuracy': fa,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'y_true': y_true, 'y_scores': y_scores, 'confusion_matrix': cm
    }

def print_metrics(metrics):
    print(f"\n{'='*60}")
    print(f"EVALUATION RESULTS")
    print(f"{'='*60}")
    print(f"Overall Accuracy:    {metrics['accuracy']:.2f}%  {'PASS' if metrics['accuracy'] >= 80 else 'FAIL'}")
    print(f"Equal Error Rate:    {metrics['eer']:.2f}%  {'PASS' if metrics['eer'] <= 12 else 'FAIL'}")
    print(f"F1 Score:            {metrics['f1_score']:.2f}%  {'PASS' if metrics['f1_score'] >= 80 else 'FAIL'}")
    print(f"Genuine Accuracy:    {metrics['genuine_accuracy']:.2f}%  {'PASS' if metrics['genuine_accuracy'] >= 75 else 'FAIL'}")
    print(f"Deepfake Accuracy:   {metrics['deepfake_accuracy']:.2f}%  {'PASS' if metrics['deepfake_accuracy'] >= 75 else 'FAIL'}")
    print(f"\nConfusion Matrix:")
    print(f"  TN={metrics['tn']:>6}  FP={metrics['fp']:>6}")
    print(f"  FN={metrics['fn']:>6}  TP={metrics['tp']:>6}")
    print(f"{'='*60}")
    all_pass = (metrics['accuracy'] >= 80 and metrics['eer'] <= 12 and
                metrics['f1_score'] >= 80 and metrics['genuine_accuracy'] >= 75 and metrics['deepfake_accuracy'] >= 75)
    print(f"VERIFICATION: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")


## 7. Training Loop

**Strategy**:
- Weighted CrossEntropy loss (inverse class frequency)
- Adam optimizer with ReduceLROnPlateau scheduler
- Early stopping on validation EER (patience=10)
- Data augmentation: noise, time-stretch, pitch-shift


In [ ]:
def train_model(model, train_loader, val_loader, epochs=40, lr=3e-4, patience=10):
    class_weights = train_loader.dataset.class_weights.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    best_eer = float('inf')
    best_state = None
    patience_ctr = 0
    history = {'train_loss': [], 'val_loss': [], 'val_eer': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_train_loss = train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        val_loss = 0.0
        model.eval()
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                val_loss += criterion(model(x), y).item()

        avg_val_loss = val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)
        scheduler.step(avg_val_loss)

        metrics = evaluate(model, val_loader)
        history['val_eer'].append(metrics['eer'])
        history['val_acc'].append(metrics['accuracy'])

        print(f"Epoch {epoch+1}: TrainLoss={avg_train_loss:.4f}, ValLoss={avg_val_loss:.4f}, "
              f"ValEER={metrics['eer']:.2f}%, ValAcc={metrics['accuracy']:.2f}%")

        if metrics['eer'] < best_eer:
            best_eer = metrics['eer']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
            print(f"  -> New best! EER={best_eer:.2f}%")
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    return model, history, best_eer

# Prepare data loaders
BATCH_SIZE = 64
n_train = min(15000, len(train_ds))
n_val = min(3000, len(val_ds))
rng = np.random.RandomState(42)
train_sub = Subset(train_ds, rng.choice(len(train_ds), n_train, replace=False))
val_sub = Subset(val_ds, rng.choice(len(val_ds), n_val, replace=False))

train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


## 8. Train the Model


In [ ]:
model = LCNN(input_dim=180, num_classes=2).to(DEVICE)
print(f"Training on {DEVICE}...")
start_time = time.time()

model, history, best_eer = train_model(
    model, train_loader, val_loader,
    epochs=40, lr=3e-4, patience=10
)

elapsed = (time.time() - start_time) / 60
print(f"\nTraining completed in {elapsed:.1f} minutes")
print(f"Best validation EER: {best_eer:.2f}%")

# Save model
os.makedirs('trained_models', exist_ok=True)
torch.save(model.state_dict(), 'trained_models/best_model.pth')
with open('trained_models/model_config.json', 'w') as f:
    json.dump({'arch': 'lcnn', 'input_dim': 180, 'num_classes': 2, 'best_val_eer': best_eer}, f, indent=2)
print("Model saved to trained_models/best_model.pth")


## 9. Training Curves


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['val_eer'], 'r-', label='Val EER')
ax2.plot(history['val_acc'], 'b-', label='Val Accuracy')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Percentage')
ax2.set_title('Validation Metrics'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 10. Final Evaluation on Test Set


In [ ]:
# Evaluate on test set
test_metrics = evaluate(model, test_loader)
print_metrics(test_metrics)

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(test_metrics['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Genuine', 'Deepfake'], yticklabels=['Genuine', 'Deepfake'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.show()


## 11. EER Curve


In [ ]:
def plot_eer_curve(metrics):
    y_true = metrics['y_true']
    y_scores = metrics['y_scores']
    genuine = y_scores[y_true == 0]
    spoof = y_scores[y_true == 1]

    thresholds = np.linspace(0, 1, 1000)
    far = np.array([np.sum(genuine >= t) / len(genuine) for t in thresholds])
    frr = np.array([np.sum(spoof < t) / len(spoof) for t in thresholds])
    diff = np.abs(far - frr)
    eer_idx = diff.argmin()
    eer_val = (far[eer_idx] + frr[eer_idx]) / 2

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(thresholds, far, 'b-', label='FAR (False Acceptance Rate)', linewidth=2)
    ax.plot(thresholds, frr, 'r-', label='FRR (False Rejection Rate)', linewidth=2)
    ax.plot(thresholds[eer_idx], eer_val, 'go', markersize=10, label=f'EER = {eer_val:.4f}')
    ax.axhline(y=eer_val, color='green', linestyle='--', alpha=0.5)
    ax.set_xlabel('Threshold'); ax.set_ylabel('Error Rate')
    ax.set_title('EER Curve — FAR vs FRR'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.show()

plot_eer_curve(test_metrics)


## 12. Score Distribution (Genuine vs Deepfake)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
genuine_scores = test_metrics['y_scores'][test_metrics['y_true'] == 0]
spoof_scores = test_metrics['y_scores'][test_metrics['y_true'] == 1]
ax.hist(genuine_scores, bins=50, alpha=0.6, label='Genuine (Human)', color='green', density=True)
ax.hist(spoof_scores, bins=50, alpha=0.6, label='Deepfake (AI)', color='red', density=True)
ax.axvline(x=0.5, color='black', linestyle='--', label='Decision Threshold (0.5)')
ax.set_xlabel('Model Score (confidence for Deepfake)'); ax.set_ylabel('Density')
ax.set_title('Score Distribution — Genuine vs Deepfake'); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()


## 13. Summary

| Metric | Score | Threshold | Status |
|--------|-------|-----------|--------|
| Overall Accuracy | — | ≥ 80% | — |
| Equal Error Rate | — | ≤ 12% | — |
| F1 Score | — | ≥ 80% | — |
| Genuine Accuracy | — | ≥ 75% | — |
| Deepfake Accuracy | — | ≥ 75% | — |

*(Results populated after training completes)*

### Key Architecture Decisions
- **LFCC over MFCC**: Linear frequency scale preserves high-frequency artifacts from neural vocoders
- **Weighted loss**: Compensates for class imbalance (inverse class frequency)
- **LCNN**: Compact architecture (606K params) with MFM activation for anti-spoofing
- **Data augmentation**: Improves generalization to unseen audio

### Deliverables
- `trained_models/best_model.pth` — Trained model weights
- `predict.py` — CLI script for single-audio inference
- `app/streamlit_app.py` — Interactive web application
- `reports/performance_report.md` — Full performance report
